# Does the answer change if we use the average instead of the typical tweet?

**Data used:** the experiment itself (who got a CN reply + Views/Likes/Shares)

**Short answer:** Yes, and it matters. On the typical (median) tweet the CN backfire is real. On the raw average it flips — but only because ~6 Control tweets went massively viral, and that flip isn’t statistically real.



# Averages-Based Re-Analysis

**Motivation.** Look at averages instead of the distribution — do the conclusions change?

**Why this matters.** Our engagement outcomes (Views/Likes/Shares growth) are heavily right-skewed — a handful of tweets go viral and sit far out in the tail. The original analysis used **medians + Mann–Whitney U**, which are robust to those outliers. That's the statistically conservative choice, but it deliberately *down-weights* the viral tail. The mean does the opposite: it's dominated by the tail. So the question is: **does any finding look materially different — or more interesting — through a mean-based lens, and if so, why?**

**Scope (surgical, per the task's own selection rule).** We do NOT mechanically re-run all 18 moderators on means — the multi-method moderator search already showed they're null, and re-running nulls just adds noise. We check the places where mean-vs-median genuinely matters:
1. **Main effect** (Views/Likes/Shares) — median/MWU vs mean/Welch-t, on raw AND winsorized data.
2. **Engagement-quality rates** (Likes-per-View, Shares-per-View).
3. **The one moderator** that survived the moderator search (`rhetorical_style × Shares`).
Plus a **descriptive inventory** of every prior analysis and the statistic it used (no re-run).

**Reporting rule:** flag only contrasts where the mean view is *substantively* different from the median view.


## Section 0 — Config & Imports

In [ ]:
import os
from pathlib import Path

BASE_DIR = Path.cwd()
while not (BASE_DIR / 'data').is_dir() and BASE_DIR != BASE_DIR.parent:
    BASE_DIR = BASE_DIR.parent
DATA_DIR  = BASE_DIR / 'data'
TASK_OUT  = BASE_DIR / 'outputs'
OUT_DIR   = TASK_OUT / 'median_vs_mean'
OUT_DIR.mkdir(parents=True, exist_ok=True)

CONTROL_XL    = DATA_DIR / 'Control_Group.xlsx'
TREATMENT_XL  = DATA_DIR / 'Treatment_Group.xlsx'
MONITORING_XL = DATA_DIR / 'Tweet Monitoring.xlsx'
FEATURES_CSV     = DATA_DIR / 'tweet_features.csv'

METRICS     = ['Views', 'Likes', 'Shares']
MAIN_WINDOW = 13
RANDOM_SEED = 42
N_BOOT      = 5000
print(f'Out dir: {OUT_DIR}')


In [ ]:
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
warnings.filterwarnings('ignore')
np.random.seed(RANDOM_SEED)
print('Imports OK')


In [ ]:
def parse_mixed_date(date_val):
    if pd.isna(date_val): return pd.NaT
    s = str(date_val).strip()
    def is_valid(year, month):
        return (year == 2025 and month == 12) or (year == 2026 and month == 1)
    if '-' in s and s[:4].isdigit():
        parts = s.split('-')
        year = int(parts[0]); n1 = int(parts[1]); n2 = int(parts[2].split()[0])
        if is_valid(year, n1):    month, day = n1, n2
        elif is_valid(year, n2):  month, day = n2, n1
        else:                      month, day = n1, n2
        return pd.Timestamp(year=year, month=month, day=day)
    elif '/' in s:
        parts = s.split('/')
        n1 = int(parts[0]); n2 = int(parts[1]); year = int(parts[2].split()[0])
        if is_valid(year, n2):    day, month = n1, n2
        elif is_valid(year, n1):  month, day = n1, n2
        else:                      day, month = n1, n2
        return pd.Timestamp(year=year, month=month, day=day)
    return pd.to_datetime(date_val, errors='coerce')
print('Date parser ready.')


## Section 1 — Build Outcomes (raw + winsorized) and Engagement Rates

Build three versions of each growth outcome so we can see the outlier influence explicitly:
- **raw growth** = (Day13 − Day0) / (Day0 + 1) × 100  (no winsorizing — the tail is fully present)
- **winsorized growth** = raw clipped at the 99th percentile (the convention used everywhere else)

Plus engagement-quality rates at day 13: Likes-per-View and Shares-per-View.


In [ ]:
control   = pd.read_excel(CONTROL_XL)
treatment = pd.read_excel(TREATMENT_XL)
monitoring = pd.read_excel(MONITORING_XL)
monitoring['Sample_Date']           = monitoring['Sample_Date'].apply(parse_mixed_date)
monitoring['Start_Date (Creation)'] = monitoring['Start_Date (Creation)'].apply(parse_mixed_date)
monitoring['Day'] = (monitoring['Sample_Date'] - monitoring['Start_Date (Creation)']).dt.days

pivot = monitoring.pivot_table(index='URL', columns='Day',
                               values=['Views','Likes','Comments','Shares'], aggfunc='first')
pivot.columns = [f'{m}_Day{d}' for m, d in pivot.columns]
pivot = pivot.reset_index()
group_map = pd.concat([control[['URL']].assign(Group='Control'),
                       treatment[['URL']].assign(Group='Treatment')])
df = pivot.merge(group_map, on='URL', how='left')

for metric in ['Likes','Shares','Comments']:
    c = f'{metric}_Day0'
    if c in df.columns: df[c] = df[c].fillna(0)

for metric in METRICS:
    d0, dN = f'{metric}_Day0', f'{metric}_Day{MAIN_WINDOW}'
    raw = (df[dN] - df[d0]) / (df[d0] + 1) * 100
    df[f'{metric}_growth_raw'] = raw
    df[f'{metric}_growth_w']   = raw.clip(upper=raw.quantile(0.99))

# Engagement-quality rates at day 13 (per 1k views, to keep numbers readable)
df['Likes_per_kView']  = df['Likes_Day13']  / (df['Views_Day13'] + 1) * 1000
df['Shares_per_kView'] = df['Shares_Day13'] / (df['Views_Day13'] + 1) * 1000

df['is_treatment'] = (df['Group'] == 'Treatment').astype(int)
print(f'df: {df.shape}')
print(f"Groups: {df['Group'].value_counts().to_dict()}")


## Section 2 — Inventory of Median / Distribution-Based Results (descriptive, no re-run)

A documented scan of where the project used median / MWU / distribution-based statistics, and whether a mean-based view is worth checking. This satisfies the "scan everything" requirement without mechanically re-running null analyses.


In [ ]:
inventory = pd.DataFrame([
    # analysis, statistic used, mean-check worth it?, reason
    ('Main effect: Views growth',  'median + MWU',          'YES', 'Headline result; heavy right tail — mean could differ'),
    ('Main effect: Likes growth',  'median + MWU',          'YES', 'Null on median; check if mean reveals outlier-driven penalty'),
    ('Main effect: Shares growth', 'median + MWU',          'YES', 'Null on median; check mean'),
    ('Engagement rate: Likes/View','median (ratio)',        'YES', 'Ratio with skew; mean sensitive to low-view tweets'),
    ('Engagement rate: Shares/View','median (ratio)',       'YES', 'Same'),
    ('trajectory shapes',  'MWU on shape params',   'NO',  'Shape params are bounded ratios/day-indices; mean adds little'),
    ('CN relevance',        'quartile MWU + reg',    'NO',  'moderator synthesis: null across methods'),
    ('poster features',     'interaction reg',       'NO',  'moderator synthesis: null across methods'),
    ('CN content',          'stratified MWU',        'NO',  'moderator synthesis: null across methods'),
    ('tweet features',      'interaction reg',       'PART','Only rhetorical_style×Shares survived the moderator synthesis — check that one'),
    ('Podolak',            'engagement ratios',     'NO',  'Already ratio-based; covered by rate check above'),
    ('Saveski',            'CATE distribution',     'NO',  'Distribution IS the point; mean not meaningful'),
    ('causal forest BLP',  'OLS on CATEs',          'NO',  'Already mean-based (OLS)'),
    ('GPI',                'reg p-values',          'NO',  'Already mean-based (OLS)'),
    ('power',             'MWU power sim',         'NO',  'Method-specific; not a descriptive contrast'),
], columns=['analysis','statistic_used','mean_check','reason'])

inventory.to_csv(OUT_DIR / 'inventory_median_based.csv', index=False)
print('=== INVENTORY: where mean-based re-analysis is worth checking ===')
print(inventory.to_string(index=False))
print()
print(f"Selected for mean re-analysis: {(inventory['mean_check']!='NO').sum()} of {len(inventory)} analyses")


## Section 3 — Main Effect: Median vs Mean (raw + winsorized)

For each outcome, report side by side:
- **Median view:** median(T), median(C), median difference, MWU p, rank-biserial r
- **Mean view:** mean(T), mean(C), mean difference, Welch t-test p, Cohen's d
- **Context:** skew of each group, mean/median gap (how much the tail pulls the mean)

We do this on **raw** growth (full tail) and **winsorized** growth (tail clipped at p99) so the outlier influence is explicit.


In [ ]:
def compare_median_mean(t_vals, c_vals):
    t = np.asarray(t_vals, float); t = t[~np.isnan(t)]
    c = np.asarray(c_vals, float); c = c[~np.isnan(c)]
    # median / MWU
    u, p_mwu = stats.mannwhitneyu(t, c, alternative='two-sided')
    r = 1 - (2*u)/(len(t)*len(c))
    med_t, med_c = np.median(t), np.median(c)
    # mean / Welch t
    tstat, p_t = stats.ttest_ind(t, c, equal_var=False)
    mean_t, mean_c = np.mean(t), np.mean(c)
    # Cohen's d (pooled SD)
    sp = np.sqrt(((len(t)-1)*np.var(t,ddof=1) + (len(c)-1)*np.var(c,ddof=1)) / (len(t)+len(c)-2))
    d = (mean_t - mean_c) / sp if sp > 0 else np.nan
    return dict(
        n_t=len(t), n_c=len(c),
        median_T=med_t, median_C=med_c, median_diff=med_t-med_c, mwu_p=p_mwu, mwu_r=r,
        mean_T=mean_t, mean_C=mean_c, mean_diff=mean_t-mean_c, welch_p=p_t, cohen_d=d,
        skew_T=stats.skew(t), skew_C=stats.skew(c),
        mean_med_gap_T=mean_t-med_t, mean_med_gap_C=mean_c-med_c,
    )

rows = []
for scale in ['raw', 'w']:
    col_suffix = 'growth_raw' if scale == 'raw' else 'growth_w'
    for metric in METRICS:
        col = f'{metric}_{col_suffix}'
        t_vals = df.loc[df['is_treatment']==1, col].values
        c_vals = df.loc[df['is_treatment']==0, col].values
        res = compare_median_mean(t_vals, c_vals)
        res.update(dict(outcome=metric, scale=('raw' if scale=='raw' else 'winsorized')))
        rows.append(res)

main_cmp = pd.DataFrame(rows)
order = ['outcome','scale','n_t','n_c','median_diff','mwu_p','mwu_r',
         'mean_diff','welch_p','cohen_d','skew_T','skew_C']
main_cmp = main_cmp[order + [c for c in main_cmp.columns if c not in order]]
main_cmp.to_csv(OUT_DIR / 'main_effect_median_vs_mean.csv', index=False)

pd.set_option('display.width', 200); pd.set_option('display.max_columns', None)
print('=== MAIN EFFECT: median vs mean ===')
print(main_cmp[order].round(4).to_string(index=False))


## Section 4 — Engagement-Quality Rates: Median vs Mean

Likes-per-1k-Views and Shares-per-1k-Views at day 13. The original analysis hinted Treatment had slightly lower engagement *quality* (not significant). Does the mean tell a sharper story (e.g., viral Treatment tweets pulling lots of low-engagement views)?


In [ ]:
rate_rows = []
for rate_col, label in [('Likes_per_kView','Likes per 1k Views'),
                        ('Shares_per_kView','Shares per 1k Views')]:
    t_vals = df.loc[df['is_treatment']==1, rate_col].values
    c_vals = df.loc[df['is_treatment']==0, rate_col].values
    res = compare_median_mean(t_vals, c_vals)
    res.update(dict(outcome=label, scale='rate'))
    rate_rows.append(res)

rate_cmp = pd.DataFrame(rate_rows)
order = ['outcome','n_t','n_c','median_T','median_C','median_diff','mwu_p',
         'mean_T','mean_C','mean_diff','welch_p','cohen_d']
rate_cmp[order].round(4)
rate_cmp.to_csv(OUT_DIR / 'engagement_rates_median_vs_mean.csv', index=False)
print('=== ENGAGEMENT RATES: median vs mean ===')
print(rate_cmp[order].round(4).to_string(index=False))


## Section 5 — The One Moderator: rhetorical_style × Shares (median vs mean)

The multi-method moderator search's single consistent winner. The original test was an interaction regression (mean-based already), but the *forest plots* stratified by rhetorical_style used medians. Here we compare, within each rhetorical_style category, the Treatment−Control difference on Shares growth, median vs mean.


In [ ]:
feat = pd.read_csv(FEATURES_CSV)[['URL','rhetorical_style']]
dfr = df.merge(feat, on='URL', how='left')

rows = []
for style in sorted(dfr['rhetorical_style'].dropna().unique()):
    sub = dfr[dfr['rhetorical_style'] == style]
    t_vals = sub.loc[sub['is_treatment']==1, 'Shares_growth_w'].values
    c_vals = sub.loc[sub['is_treatment']==0, 'Shares_growth_w'].values
    if len(t_vals) < 10 or len(c_vals) < 10:
        rows.append(dict(rhetorical_style=style, n_t=len(t_vals), n_c=len(c_vals),
                         note='n<10, skipped'))
        continue
    res = compare_median_mean(t_vals, c_vals)
    res['rhetorical_style'] = style
    rows.append(res)

rstyle_cmp = pd.DataFrame(rows)
cols = ['rhetorical_style','n_t','n_c','median_diff','mwu_p','mean_diff','welch_p','cohen_d']
cols = [c for c in cols if c in rstyle_cmp.columns]
rstyle_cmp.to_csv(OUT_DIR / 'rhetorical_style_shares_median_vs_mean.csv', index=False)
print('=== rhetorical_style × Shares: T-C difference by style, median vs mean ===')
print(rstyle_cmp[cols].round(4).to_string(index=False))
print()
print('(Sign convention: positive diff = Treatment higher = anti-suppression;')
print(' negative = suppression. The moderator-synthesis finding is narrative-storytelling flips to suppression.)')


## Section 6 — Selection: Which Contrasts Change Under the Mean View?

Flag a contrast as **informative** if the mean view tells a *substantively different* story from the median view — operationalized as either:
- a **sign flip** (median diff and mean diff point in opposite directions), or
- a **significance flip** (significant under one statistic, not the other, at α=0.05), or
- a large **mean/median gap** relative to the effect (tail is doing the work).


In [ ]:
def classify(row):
    flags = []
    md_, mn_ = row.get('median_diff', np.nan), row.get('mean_diff', np.nan)
    pmwu, pt = row.get('mwu_p', np.nan), row.get('welch_p', np.nan)
    if pd.notna(md_) and pd.notna(mn_) and np.sign(md_) != np.sign(mn_) and abs(md_) > 1e-9:
        flags.append('SIGN FLIP')
    if pd.notna(pmwu) and pd.notna(pt):
        sig_mwu, sig_t = pmwu < 0.05, pt < 0.05
        if sig_mwu != sig_t:
            flags.append(f'SIG FLIP (MWU {"sig" if sig_mwu else "ns"} / t {"sig" if sig_t else "ns"})')
    return '; '.join(flags) if flags else 'consistent'

# Apply to main effect + rates
flagged = []
for _, r in main_cmp.iterrows():
    flagged.append(dict(contrast=f"{r['outcome']} growth ({r['scale']})",
                        median_diff=r['median_diff'], mean_diff=r['mean_diff'],
                        mwu_p=r['mwu_p'], welch_p=r['welch_p'], verdict=classify(r)))
for _, r in rate_cmp.iterrows():
    flagged.append(dict(contrast=r['outcome'],
                        median_diff=r['median_diff'], mean_diff=r['mean_diff'],
                        mwu_p=r['mwu_p'], welch_p=r['welch_p'], verdict=classify(r)))

flag_df = pd.DataFrame(flagged)
flag_df.to_csv(OUT_DIR / 'selection_flags.csv', index=False)
print('=== SELECTION: which contrasts change under the mean view? ===')
print(flag_df.round(4).to_string(index=False))
print()
informative = flag_df[flag_df['verdict'] != 'consistent']
print(f'Informative (mean view differs): {len(informative)} of {len(flag_df)} contrasts')


## Section 7 — Visualization + Narrative

Dumbbell plot: for each main-effect outcome, show median diff vs mean diff (winsorized), so the gap between the two views is visible at a glance.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5), dpi=150)
sub = main_cmp[main_cmp['scale']=='winsorized'].reset_index(drop=True)
y = np.arange(len(sub))
for i, r in sub.iterrows():
    ax.plot([r['median_diff'], r['mean_diff']], [i, i], '-', color='#bbbbbb', lw=2, zorder=1)
ax.scatter(sub['median_diff'], y, s=110, color='#1f77b4', label='Median diff (MWU)', zorder=3)
ax.scatter(sub['mean_diff'],   y, s=110, color='#d62728', label='Mean diff (Welch t)', zorder=3)
ax.axvline(0, color='black', lw=0.8)
ax.set_yticks(y); ax.set_yticklabels(sub['outcome'])
ax.set_xlabel('Treatment − Control difference (winsorized growth %)')
ax.set_title('Median vs Mean view of the main effect\n(gap = how much the viral right-tail pulls the mean)')
ax.legend(); ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig_median_vs_mean_dumbbell.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved: fig_median_vs_mean_dumbbell.png')


In [ ]:
# Auto-generated narrative scaffold (fill specifics from the printed tables)
print('=' * 64)
print('NARRATIVE SCAFFOLD — fill in from tables above')
print('=' * 64)
print('''
1. MAIN EFFECT
   - Views: does the anti-suppression hold on the mean? Compare median_diff vs
     mean_diff (raw and winsorized). Larger mean gap = viral Treatment tweets.
   - Likes/Shares: median said null. Does the mean reveal an outlier-driven
     pattern, or stay null? (Check welch_p and sign.)

2. RAW vs WINSORIZED
   - If a finding only appears on RAW means (vanishes when winsorized), it is
     entirely outlier-driven — report with caution, do not headline.

3. ENGAGEMENT RATES
   - Does the mean sharpen the "Treatment lowers engagement quality" hint?

4. rhetorical_style x Shares
   - Does narrative-storytelling still flip to suppression on the mean?

5. BOTTOM LINE for the paper
   - Median-based results are primary (robust to tail). Mean-based views are
     reported only where flagged INFORMATIVE in Section 6, with an explicit
     note that the difference is tail-driven.
''')


## Section 8 — Outputs Index

Saved to `outputs/02_median_vs_mean/`:
- `inventory_median_based.csv` — scan of all prior analyses + whether a mean check was warranted.
- `main_effect_median_vs_mean.csv` — main effect, both raw and winsorized, median vs mean.
- `engagement_rates_median_vs_mean.csv` — Likes/Shares per-view rates.
- `rhetorical_style_shares_median_vs_mean.csv` — the one moderator, by style.
- `selection_flags.csv` — which contrasts change under the mean view (sign/significance flips).
- `fig_median_vs_mean_dumbbell.png` — visual of the median-vs-mean gap.

**Reading guide:** medians stay primary (robust to the viral tail). Mean-based numbers are reported only for contrasts flagged *informative* in Section 6, always with the note that the difference is driven by the right tail.
